# Indian Air Quality Intelligence Platform

## Notebook 03 – Data Cleaning

### Objectives

- Create a working copy of the raw dataset
- Remove columns with 100% missing values
- Remove duplicate records
- Handle invalid pollutant values
- Handle remaining missing values
- Optimize data types
- Save the cleaned dataset

**Output**

data/processed/aqi_cleaned.csv

reports/tables/data_cleaning_report.csv


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
DATA_PATH = Path("../data/raw/INDIA_AQI_COMPLETE_20251126.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} not found.")

df = pd.read_csv(DATA_PATH)

print("✅ Dataset Loaded Successfully")

✅ Dataset Loaded Successfully


In [3]:
df_clean = df.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [4]:
print("="*50)
print("Initial Dataset Shape")
print("="*50)

print(df_clean.shape)

Initial Dataset Shape
(842160, 71)


In [5]:
missing_percent = df_clean.isnull().mean()*100

cols_to_drop = missing_percent[
    missing_percent==100
].index.tolist()

print(f"Columns to Drop : {len(cols_to_drop)}")

cols_to_drop

Columns to Drop : 8


['Temp_80m_C',
 'Temp_120m_C',
 'Temp_180m_C',
 'Wind_Speed_80m_kmh',
 'Wind_Speed_120m_kmh',
 'UV_Index',
 'NH3_ugm3',
 'Inversion_Strength_C']

In [6]:
df_clean.drop(
    columns=cols_to_drop,
    inplace=True
)

print("Columns removed successfully.")

Columns removed successfully.


In [7]:
print("="*50)
print("Dataset Shape After Removing Empty Columns")
print("="*50)

print(df_clean.shape)

Dataset Shape After Removing Empty Columns
(842160, 63)


In [9]:
duplicates = df_clean.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 0


In [12]:
pollutant_columns = [
    "PM2_5_ugm3",
    "PM10_ugm3",
    "NO2_ugm3",
    "SO2_ugm3",
    "CO_ugm3",
    "O3_ugm3",
    "NH3_ugm3"
]

negative_summary = {}

for col in pollutant_columns:
    if col in df_clean.columns:
        negative_summary[col] = (df_clean[col] < 0).sum()

negative_df = (
    pd.DataFrame.from_dict(
        negative_summary,
        orient="index",
        columns=["Negative Values"]
    )
    .sort_values("Negative Values", ascending=False)
)

negative_df

,Negative Values
O3_ugm3,82
SO2_ugm3,4
NO2_ugm3,2
PM2_5_ugm3,0
PM10_ugm3,0
CO_ugm3,0


In [27]:
df_clean.sample(1)

,City,State,Latitude,Longitude,Datetime,Year,Month,Day,Hour,Day_of_Week,Day_Name,Week_of_Year,Is_Weekend,Quarter,Season,Time_of_Day,Temp_2m_C,Humidity_Percent,Dew_Point_C,Humidity_Category,Wind_Speed_10m_kmh,Wind_Dir_10m,Wind_Gusts_kmh,Wind_Category,Wind_Stagnation,Precipitation_mm,Rain_mm,Is_Raining,Heavy_Rain,Pressure_MSL_hPa,Surface_Pressure_hPa,Solar_Radiation_Wm2,Direct_Radiation_Wm2,Diffuse_Radiation_Wm2,Cloud_Cover_Percent,Cloud_Low_Percent,Cloud_Mid_Percent,Cloud_High_Percent,Is_Daytime,Sunshine_Seconds,PM2_5_ugm3,PM10_ugm3,PM_Ratio,CO_ugm3,NO2_ugm3,SO2_ugm3,O3_ugm3,Dust_ugm3,AOD,US_AQI,US_AQI_PM25,US_AQI_PM10,US_AQI_NO2,US_AQI_O3,US_AQI_CO,EU_AQI,EU_AQI_PM25,EU_AQI_PM10,AQI_Category,PM25_Category_India,Temp_Inversion,Festival_Period,Crop_Burning_Season
168261,Bhubaneswar,Odisha,20.30,85.82,2025-03-22 21:00:00,2025,3,22,21,5,Saturday,12,1,1,Summer,Night_Late,22.00,89,20.10,Very_Humid,7.10,207,17.30,Moderate,0,0.00,0.00,0,0,1015.00,1009.80,0.00,0.00,0.00,100,0,77,100,0,0.00,65.20,69.60,0.94,375.00,39.40,15.80,31.00,4.00,0.45,81.00,81.00,27.00,19.00,36.00,3,61.00,61.00,30.00,Moderate,Moderate,0,0,0


In [18]:
for col in pollutant_columns:
    if col in df_clean.columns:
        df_clean.loc[df_clean[col] < 0, col] = np.nan

print("✅ Negative pollutant values converted to NaN.")

✅ Negative pollutant values converted to NaN.


In [19]:
negative_after = {}

for col in pollutant_columns:
    if col in df_clean.columns:
        negative_after[col] = (df_clean[col] < 0).sum()

pd.DataFrame.from_dict(
    negative_after,
    orient="index",
    columns=["Negative Values After Cleaning"]
)

,Negative Values After Cleaning
PM2_5_ugm3,0
PM10_ugm3,0
NO2_ugm3,0
SO2_ugm3,0
CO_ugm3,0
O3_ugm3,0


In [20]:
missing = pd.DataFrame({
    "Missing Values": df_clean.isnull().sum(),
    "Missing %": (df_clean.isnull().mean() * 100).round(3)
})

missing = missing[missing["Missing Values"] > 0]

missing.sort_values("Missing Values", ascending=False)

,Missing Values,Missing %
AQI_Category,2516,0.30
US_AQI,145,0.02
US_AQI_PM25,145,0.02
US_AQI_PM10,145,0.02
EU_AQI,145,0.02
EU_AQI_PM25,145,0.02
EU_AQI_PM10,145,0.02
O3_ugm3,82,0.01
US_AQI_O3,73,0.01
SO2_ugm3,4,0.00


In [23]:
df_clean["Datetime"] = pd.to_datetime(df_clean["Datetime"])

In [24]:
df_clean.dtypes

City                           object
State                          object
Latitude                      float64
Longitude                     float64
Datetime               datetime64[ns]
                            ...      
AQI_Category                   object
PM25_Category_India            object
Temp_Inversion                  int64
Festival_Period                 int64
Crop_Burning_Season             int64
Length: 63, dtype: object

In [25]:
from pathlib import Path

OUTPUT_PATH = Path("../data/processed/aqi_cleaned.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Cleaned dataset saved to: {OUTPUT_PATH}")

✅ Cleaned dataset saved to: ..\data\processed\aqi_cleaned.csv


In [29]:
saved_df = pd.read_csv(OUTPUT_PATH)

print("Saved Shape :", saved_df.shape)

Saved Shape : (842160, 63)


In [32]:
cleaning_report = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Original Columns",
        "Final Rows",
        "Final Columns",
        "Columns Removed",
        "Duplicate Rows Removed",
        "Negative Values Fixed"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df_clean.shape[0],
        df_clean.shape[1],
        len(cols_to_drop),
        duplicates,
        88
    ]
})

cleaning_report

,Metric,Value
0,Original Rows,842160
1,Original Columns,71
2,Final Rows,842160
3,Final Columns,63
4,Columns Removed,8
5,Duplicate Rows Removed,0
6,Negative Values Fixed,88


In [33]:
cleaning_report.to_csv(
    "../reports/tables/data_cleaning_report.csv",
    index=False
)